# Lesson 08: Testing & Debugging MCP Servers

In this notebook, you will:
1. Write async integration tests using `mcp.shared.memory.create_connected_server_and_client_session`.
2. Verify tool schemas, annotations, and error handling.
3. Run a Python Inspector diagnostic dump over a `FastMCP` server.

In [ ]:
!pip install -q -U "mcp<2" pytest pytest-asyncio

In [ ]:
import asyncio
from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session

mcp = FastMCP("TestableServer")

@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ValueError("Division by zero")
    return a / b

async def run_inspector_checks():
    async with create_connected_server_and_client_session(mcp._mcp_server) as session:
        tools = await session.list_tools()
        assert len(tools.tools) == 1
        assert tools.tools[0].name == "divide"

        ok = await session.call_tool("divide", {"a": 10, "b": 2})
        assert float(ok.content[0].text) == 5.0

        err = await session.call_tool("divide", {"a": 10, "b": 0})
        assert err.isError is True
        print("All 3 in-memory Inspector & conformance checks passed!")

await run_inspector_checks()

### Challenge Exercise
Write a `test_mcp_server.py` file using `%%writefile` with two `pytest` async tests and run `!pytest -v test_mcp_server.py`.